### Task 1 – Download Mobile Phone Dataset and Train a Model

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
df = pd.read_csv("mobile_price.csv")

print("First 5 Rows:")
display(df.head())

print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

First 5 Rows:


,brand,model,launch_price_inr,launch_year,age_years,city,seller_type,ram_gb,storage_gb,processor,...,condition,screen_crack,scratches,box,charger,invoice,warranty,network,repair_history,estimated_resale_price_inr
0,Apple,iPhone 11,49999,2022,4,Pune,Individual,12,64,Dimensity 9200,...,Good,No,NaN,Yes,Yes,Yes,No,5G,NaN,17800
1,OnePlus,Nord 3,33999,2025,1,Delhi,Individual,6,256,Snapdragon 695,...,Good,No,Moderate,No,Yes,No,No,5G,NaN,21300
2,Realme,11 Pro+,32999,2026,0,Noida,Individual,12,256,A16 Bionic,...,Good,No,Moderate,Yes,No,No,Yes,5G,NaN,22700
3,Xiaomi,Redmi Note 12 Pro,25999,2023,3,Kolkata,Individual,8,128,Snapdragon 7 Gen 3,...,Excellent,No,NaN,Yes,Yes,No,No,5G,NaN,11300
4,iQOO,Z9,21999,2023,3,Patna,Individual,6,512,Dimensity 8200,...,Excellent,No,Minor,Yes,Yes,Yes,No,5G,NaN,9100


Dataset Shape: (3000, 23)

Column Names:
['brand', 'model', 'launch_price_inr', 'launch_year', 'age_years', 'city', 'seller_type', 'ram_gb', 'storage_gb', 'processor', 'display_hz', 'battery_health_pct', 'camera_mp', 'condition', 'screen_crack', 'scratches', 'box', 'charger', 'invoice', 'warranty', 'network', 'repair_history', 'estimated_resale_price_inr']


In [3]:
# Select input features
X = df[["ram_gb", "storage_gb", "battery_health_pct", "camera_mp"]]

# Select target variable
y = df["estimated_resale_price_inr"]

# Remove missing values
data = pd.concat([X, y], axis=1).dropna()

X = data[["ram_gb", "storage_gb", "battery_health_pct", "camera_mp"]]
y = data["estimated_resale_price_inr"]

print("Features:")
display(X.head())

print("\nTarget:")
display(y.head())

Features:


,ram_gb,storage_gb,battery_health_pct,camera_mp
0,12,64,74,50
1,6,256,91,108
2,12,256,98,108
3,8,128,74,64
4,6,512,79,108



Target:


0    17800
1    21300
2    22700
3    11300
4     9100
Name: estimated_resale_price_inr, dtype: int64

In [4]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Create Multiple Linear Regression model
model = LinearRegression()

# Train model
model.fit(X_train, y_train)

# Predict mobile prices
y_pred = model.predict(X_test)

print("Model Training Completed!")
print("Training Rows:", len(X_train))
print("Testing Rows:", len(X_test))

Model Training Completed!
Training Rows: 2400
Testing Rows: 600


### Task 2 – Display Regression Coefficients and Intercept

In [5]:
# Display regression coefficients
print("Regression Coefficients:")

for feature, coefficient in zip(X.columns, model.coef_):
    print(feature, ":", coefficient)

# Display intercept
print("\nIntercept:", model.intercept_)

Regression Coefficients:
ram_gb : 29.73844363463001
storage_gb : 0.39157525456089093
battery_health_pct : 581.3331328589759
camera_mp : -3.3909975768791116

Intercept: -26997.137260686817


### Task 3 – Calculate Variance Inflation Factor (VIF)

In [6]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Calculate VIF for each feature
vif_data = pd.DataFrame()

vif_data["Feature"] = X.columns

vif_data["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

print("Variance Inflation Factor:")
display(vif_data)

Variance Inflation Factor:


,Feature,VIF
0,ram_gb,7.950584
1,storage_gb,3.835966
2,battery_health_pct,12.381180
3,camera_mp,3.556946


### Task 4 – Identify the Most Important Feature

In [8]:
from sklearn.preprocessing import StandardScaler

# Standardize input features
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# Train regression model on scaled features
scaled_model = LinearRegression()
scaled_model.fit(X_scaled, y)

# Display feature importance using absolute coefficients
importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": scaled_model.coef_,
    "Importance": np.abs(scaled_model.coef_)
})

importance = importance.sort_values(
    by="Importance", ascending=False
)

print("Feature Importance:")
display(importance)

Feature Importance:


,Feature,Coefficient,Importance
2,battery_health_pct,4851.191087,4851.191087
3,camera_mp,-120.106152,120.106152
1,storage_gb,5.127248,5.127248
0,ram_gb,-2.854780,2.854780


### Task 5 – Remove a Highly Collinear Feature and Retrain Model

In [9]:
# Find feature with highest VIF
highest_vif_feature = vif_data.sort_values(
    by="VIF", ascending=False
).iloc[0]["Feature"]

print("Feature with Highest VIF:", highest_vif_feature)

# Remove the selected feature
X_new = X.drop(columns=[highest_vif_feature])

print("\nRemaining Features:")
print(X_new.columns.tolist())

Feature with Highest VIF: battery_health_pct

Remaining Features:
['ram_gb', 'storage_gb', 'camera_mp']


In [10]:
# Split new dataset
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(
    X_new, y, test_size=0.20, random_state=42
)

# Create new regression model
new_model = LinearRegression()

# Train new model
new_model.fit(X_train_new, y_train_new)

# Make predictions
new_pred = new_model.predict(X_test_new)

print("New Model Training Completed!")

print("\nNew Regression Coefficients:")
for feature, coefficient in zip(X_new.columns, new_model.coef_):
    print(feature, ":", coefficient)

print("\nNew Intercept:", new_model.intercept_)

New Model Training Completed!

New Regression Coefficients:
ram_gb : -25.699069493009084
storage_gb : 0.38938054795897864
camera_mp : -2.9968634359483364

New Intercept: 21238.307173895537


In [11]:
# Calculate old model performance
old_mse = mean_squared_error(y_test, y_pred)
old_r2 = r2_score(y_test, y_pred)

# Calculate new model performance
new_mse = mean_squared_error(y_test_new, new_pred)
new_r2 = r2_score(y_test_new, new_pred)

# Display comparison
comparison = pd.DataFrame({
    "Model": ["Original Model", "New Model"],
    "MSE": [old_mse, new_mse],
    "R2 Score": [old_r2, new_r2]
})

display(comparison)

,Model,MSE,R2 Score
0,Original Model,1.239119e+08,0.153547
1,New Model,1.468119e+08,-0.002885
